# TrainLens: explain the data, diagnose a fine-tuning run, and plan the next one

This quickstart uses realistic train/validation/test metrics plus a small synthetic dataset. It shows how TrainLens explains aggregate dataset context, plots learning curves, runs deterministic training analysis, and optionally asks an OpenAI-compatible LLM to explain the result and propose controlled next experiments.

Everything before the optional LLM section runs locally. The example data is deliberately small so the notebook runs immediately; in a real notebook, pass the dataset, metrics, model, and trainer objects you already have in memory.

In [ ]:
%pip install -q "trainlens[plots]"

## 1. Start from normal fine-tuning results

TrainLens keeps training, validation, and held-out test metrics separate. It understands common aliases such as `eval_loss`, `val_loss`, `test_f1`, and split metric dictionaries.

In [ ]:
history = {
    "train_loss":     [1.31, 0.93, 0.66, 0.47, 0.34, 0.25],
    "val_loss":       [1.28, 0.91, 0.70, 0.58, 0.55, 0.57],
    "train_accuracy": [0.55, 0.68, 0.79, 0.87, 0.92, 0.95],
    "val_accuracy":   [0.54, 0.66, 0.75, 0.80, 0.82, 0.81],
}

test_metrics = {
    "loss": 0.64,
    "accuracy": 0.75,
    "f1": 0.74,
    "precision": 0.77,
    "recall": 0.72,
}

## 2. Explain the dataset behind those metrics

`explain_dataset()` produces local, aggregate evidence about the data: split sizes, feature kinds, missingness, cardinality, numeric/text summaries, target balance, and cross-split differences. It does not include raw feature rows in the rendered profile and it does not claim that a dataset property caused a model result.

In [ ]:
train_dataset = {
    "text": [
        "short positive example",
        "another positive training example",
        "positive sample with more context",
        "negative example",
        "positive sample",
        "positive example with detail",
        "positive observation",
        "negative sample with detail",
    ],
    "source": ["web", "web", "web", "human", "web", "web", "human", "human"],
    "label": [1, 1, 1, 0, 1, 1, 1, 0],
}

validation_dataset = {
    "text": ["validation a", "validation b", "validation c", "validation d"],
    "source": ["web", "human", "human", "human"],
    "label": [1, 0, 0, 1],
}

test_dataset = {
    "text": ["test a", "test b", "test c", "test d"],
    "source": ["human", "human", "web", "human"],
    "label": [0, 0, 1, 0],
}

In [ ]:
from trainlens import explain_dataset

dataset_context = explain_dataset(
    {
        "train": train_dataset,
        "validation": validation_dataset,
        "test": test_dataset,
    },
    target="label",
)
dataset_context

In this synthetic example, the train split is target-imbalanced and the target mix changes across validation/test. Those observations are useful context for interpreting generalization, but they are descriptive evidence rather than proof that the data caused any metric gap.

## 3. Plot the run and analyze it locally

The plot is optional (`trainlens[plots]`). The training analysis itself does not call any external service.

In [ ]:
from trainlens import analyze, plot_training_curves, render_report

analysis = analyze(globals())
plot_training_curves(globals())
print(render_report(analysis))

### 3a. Visualize 1,000 recorded steps without losing anomalies

For long training jobs, TrainLens can show a plot-ready *min/max envelope* with fewer drawn points while keeping the complete local history unchanged. It also marks the best validation observation. The model-facing context is a separate compact sketch: it does **not** contain all 1,000 observations. Recorded steps are not guessed from observation indices.

This demo keeps its synthetic training logs inside a function so they do not override the real `history` and `test_metrics` variables used in subsequent notebook cells.

In [ ]:
def demo_long_training_plot():
    from math import sin
    from trainlens import plot_training_curves, preview_notebook_context

    logs = [
        {
            "step": (i + 1) * 25,
            "loss": 1.1 / (1 + 0.013 * i),
            "eval_loss": 0.95 / (1 + 0.010 * i) + 0.00018 * i + 0.01 * sin(i / 28),
        }
        for i in range(1000)
    ]
    logs[397]["eval_loss"] += 0.24  # A short anomaly worth preserving.
    run_data = {"log_history": logs}
    figure = plot_training_curves(
        run_data, metrics=("loss",), x_axis="step", max_plot_points=300
    )
    preview = preview_notebook_context(run_data)
    print("Compact LLM/agent metric evidence (not the full 1,000-row log):")
    print("\n".join(
        line for line in preview.markdown.splitlines()
        if line.startswith("- `loss`") or line.startswith("- `validation_loss`")
    ))
    return figure

demo_long_training_plot()

For this run, TrainLens should preserve the final test F1/precision/recall, notice the train/validation generalization gap, and notice that the held-out test result is weaker than validation. With only validation metrics and no test metrics, it can instead recommend a final held-out evaluation.

On real fine-tuning objects, TrainLens can additionally report trainable/frozen parameters, PEFT adapters, LoRA rank/alpha/target modules, quantization, learning rates, and supported multimodal fine-tuning settings.

## 4. Optional: connect an LLM

Use `getpass()` so the key is not written into the notebook. Replace the base URL and model with any OpenAI-compatible provider you use. For a local Ollama/LM Studio/vLLM/llama.cpp endpoint, you can usually omit the API key.

In [ ]:
from getpass import getpass

from trainlens import OpenAICompatibleProvider

LLM_BASE_URL = "https://your-provider.example/v1"
LLM_MODEL = "your-model"

llm = OpenAICompatibleProvider.from_values(
    base_url=LLM_BASE_URL,
    model=LLM_MODEL,
    api_key=getpass("LLM API key: "),
)

### Define the analysis you want

The prompt below asks the LLM to explain the observed training behavior, keep test data as final evidence rather than a tuning target, use dataset observations as context rather than causal proof, and propose controlled next experiments.

In [ ]:
from trainlens import improvement_plan_prompt, prompt_text

prompt = improvement_plan_prompt(
    objective=prompt_text("""
        Explain what happened during this fine-tuning run and propose the three
        highest-value next experiments.

        Use the test split only as final generalization evidence; do not tune to it.
        Use dataset properties as context, not as proof of causation.
        Prefer controlled, low-cost changes first.
    """),
    model_family="LLM fine-tuning with PEFT/LoRA",
    focus_areas=(
        "train/validation generalization gap",
        "validation-to-test degradation",
        "dataset imbalance or split differences",
        "checkpoint selection and early stopping",
        "learning-rate schedule",
        "regularization and adapter capacity",
    ),
)

### Preview the exact request before sending it

This performs no network request. `system_prompt` contains TrainLens' trusted instructions; `user_prompt` contains sanitized notebook evidence plus the aggregate dataset profile that you explicitly supplied. Raw feature rows are not silently attached.

In [ ]:
from trainlens import preview_llm_request

preview = preview_llm_request(
    globals(),
    mode="improvement_ideas",
    provider=llm,
    prompt_options=prompt,
    dataset_explanation=dataset_context,
)

print(preview.system_prompt)
print("\n--- SANITIZED EVIDENCE ---\n")
print(preview.user_prompt)

### Generate the explanation and next-step plan

The LLM receives deterministic TrainLens findings together with the aggregate dataset context. The improvement recipe asks it to return evidence-backed changes, expected effects, cost/risk, confidence, measurable success criteria, and one concrete next run.

In [ ]:
from trainlens import build_improvement_ideas

report = build_improvement_ideas(
    globals(),
    provider=llm,
    prompt_options=prompt,
    dataset_explanation=dataset_context,
)
report

### Optional: require machine-checkable evidence citations

`build_verified_improvement_plan()` gives model findings and dataset facts stable evidence IDs. Citations invented by the LLM are preserved as unsupported instead of being silently accepted.

In [ ]:
from trainlens import build_verified_improvement_plan

verified_plan = build_verified_improvement_plan(
    globals(),
    provider=llm,
    dataset_explanation=dataset_context,
)
print(verified_plan.recommendations)
print(verified_plan.unsupported_evidence_ids)

## Local LLM alternative

If you run an OpenAI-compatible local server, replace the provider cell with:

```python
llm = OpenAICompatibleProvider.from_values(
    base_url="http://localhost:11434/v1",
    model="your-local-model",
)
```

Everything else in the notebook stays the same.